دفترچه یادداشت زیر به طور خودکار توسط مکالمه GitHub Copilot تولید شده و فقط برای راه‌اندازی اولیه است


# مقدمه‌ای بر مهندسی دستور
مهندسی دستور فرآیند طراحی و بهینه‌سازی دستورها برای وظایف پردازش زبان طبیعی است. این شامل انتخاب دستورهای مناسب، تنظیم پارامترهای آنها و ارزیابی عملکردشان می‌شود. مهندسی دستور برای دستیابی به دقت و کارایی بالا در مدل‌های NLP بسیار حیاتی است. در این بخش، اصول مهندسی دستور را با استفاده از مدل‌های OpenAI برای کاوش بررسی خواهیم کرد.


### تمرین ۱: نشانه‌گذاری
کاوش نشانه‌گذاری با استفاده از tiktoken، یک نشانه‌گذار سریع و متن‌باز از OpenAI
برای مثال‌های بیشتر به [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst) مراجعه کنید.


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### تمرین ۲: اعتبارسنجی تنظیم کلید API اوپن‌ای‌آی

کد زیر را اجرا کنید تا بررسی شود که نقطه پایانی OpenAI شما به درستی تنظیم شده است. این کد فقط یک پرسش ساده اولیه را امتحان می‌کند و تکمیل آن را اعتبارسنجی می‌کند. ورودی `oh say can you see` باید در راستای `by the dawn's early light..` تکمیل شود.


In [ ]:
# Uses the OpenAI client against the Azure OpenAI (Microsoft Foundry) v1 endpoint
# with the Responses API. See https://aka.ms/openai/start

import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']

def get_completion(prompt):
    response = client.responses.create(
        model=deployment,
        input=prompt,
        temperature=0, # this is the degree of randomness of the model's output
        max_output_tokens=1024,
        store=False,
    )
    return response.output_text

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt)
print(response)


### تمرین ۳: ساختگی‌ها  
بررسی کنید چه اتفاقی می‌افتد وقتی از LLM بخواهید تکمیل‌هایی برای یک پرسش درباره موضوعی که ممکن است وجود نداشته باشد، یا درباره موضوعاتی که ممکن است نداند زیرا در داده‌های پیش‌آموزش‌دیده آن نبوده‌اند (جدیدتر) ارائه کند. ببینید پاسخ چگونه تغییر می‌کند اگر پرسش متفاوت یا مدل متفاوتی را امتحان کنید.  


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt)
print(response)

### تمرین ۴: مبتنی بر دستورالعمل  
از متغیر «text» برای تعیین محتوای اصلی استفاده کنید  
و از متغیر «prompt» برای ارائه یک دستورالعمل مرتبط با آن محتوای اصلی بهره ببرید.  

در اینجا از مدل می‌خواهیم که متن را برای یک دانش‌آموز کلاس دوم خلاصه کند  


In [ ]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt)
print(response)

### تمرین ۵: درخواست پیچیده  
یک درخواست را امتحان کنید که شامل پیام‌های سیستم، کاربر و دستیار باشد  
سیستم زمینه دستیار را تنظیم می‌کند  
پیام‌های کاربر و دستیار زمینه گفتگوی چندمرحله‌ای را فراهم می‌کنند  

توجه کنید که شخصیت دستیار در زمینه سیستم به صورت "طعنه‌آمیز" تنظیم شده است.  
سعی کنید از زمینه شخصیت متفاوتی استفاده کنید یا سری متفاوتی از پیام‌های ورودی/خروجی را امتحان کنید  


In [ ]:
response = client.responses.create(
    model=deployment,
    input=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ],
    store=False,
)
print(response.output_text)


### تمرین: شهود خود را کشف کنید
مثال‌های بالا الگوهایی به شما می‌دهند که می‌توانید برای ایجاد پرسش‌های جدید (ساده، پیچیده، دستورالعمل و غیره) استفاده کنید - سعی کنید تمرین‌های دیگری ایجاد کنید تا برخی از ایده‌های دیگری که درباره‌شان صحبت کردیم مانند مثال‌ها، نشانه‌ها و موارد بیشتر را کشف کنید.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
